In [3]:
import os 
os.makedirs("data/pds",exist_ok=True)

In [9]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
loader= DirectoryLoader("data/pds",
                        glob="**/*.pdf",
                        loader_cls=PyMuPDFLoader,
                        show_progress=True)
docs=loader.load()
docs

100%|██████████| 2/2 [00:01<00:00,  1.30it/s]


[Document(metadata={'producer': 'cairo 1.17.4 (https://cairographics.org)', 'creator': 'Mozilla Firefox 124.0.1', 'creationdate': '2024-04-01T12:07:18+05:30', 'source': 'data\\pds\\bns.pdf', 'file_path': 'data\\pds\\bns.pdf', 'total_pages': 102, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20240401120718+05'30", 'page': 0}, page_content='THE BHARATIYA NYAYA SANHITA, 2023\nNO. 45 OF 2023\n[25th December, 2023.]\nAn Act to consolidate and amend the provisions relating to offences and for\nmatters connected therewithor incidentalthereto.\nBE it enacted by Parliament in the Seventy-fourth Year of the Republic of India as\nfollows:––\nCHAPTERI\nPRELIMINARY\n1. (1) ThisAct maybe called the Bharatiya Nyaya Sanhita, 2023.\n(2) It shall come intoforce on such date as the Central Government may, bynotification\nin the Official Gazette, appoint, and different dates maybe appointed for different provi

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=text_splitter.split_documents(docs)
chunks

[Document(metadata={'producer': 'cairo 1.17.4 (https://cairographics.org)', 'creator': 'Mozilla Firefox 124.0.1', 'creationdate': '2024-04-01T12:07:18+05:30', 'source': 'data\\pds\\bns.pdf', 'file_path': 'data\\pds\\bns.pdf', 'total_pages': 102, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20240401120718+05'30", 'page': 0}, page_content='THE BHARATIYA NYAYA SANHITA, 2023\nNO. 45 OF 2023\n[25th December, 2023.]\nAn Act to consolidate and amend the provisions relating to offences and for\nmatters connected therewithor incidentalthereto.\nBE it enacted by Parliament in the Seventy-fourth Year of the Republic of India as\nfollows:––\nCHAPTERI\nPRELIMINARY\n1. (1) ThisAct maybe called the Bharatiya Nyaya Sanhita, 2023.\n(2) It shall come intoforce on such date as the Central Government may, bynotification\nin the Official Gazette, appoint, and different dates maybe appointed for different provi

In [ ]:
from sentence_transformers import SentenceTransformer
def conversion(text,embedd_model:str="all-MiniLM-L6-v2"):
    try:
         embedding_model=SentenceTransformer(embedd_model)

    except Exception as e:
         print(f"Embedding model failed to load : {e}")

    print(f"embedding model dimension : {embedding_model.get_embedding_dimension()}")
    embeddings=embedding_model.encode(text,show_progress_bar=True)
    return embeddings

text=[doc.page_content for doc in chunks]
embeddings=conversion(text)
embeddings.shape
    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2197.17it/s]


embedding model dimension : <bound method SentenceTransformer.get_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)>


Batches: 100%|██████████| 37/37 [01:13<00:00,  1.98s/it]


(1176, 384)

Persisting the embeddings using Chroma DB

In [18]:
import chromadb
import os

In [19]:
def chromaClient(persist_dir:str="data/chroma",collection_name="pdf_docs"):
    os.makedirs(persist_dir,exist_ok=True)
    client=chromadb.PersistentClient(path=persist_dir)
    collection=client.get_or_create_collection(name=collection_name)
    print(collection.count())
    return collection

client=chromaClient()


0


In [20]:
def embeddingAddition(client,chunks,embeddings):
    if(len(chunks)!=len(embeddings)): return 
    ids=[]
    document_content=[]
    metadatas=[]
    embeds=[]
    for i,(doc,embed) in enumerate(zip(chunks,embeddings)):
        doc_id=f"doc_#{i*11+13}_{i}"
        ids.append(doc_id)

        metadata=dict(doc.metadata)
        metadata["index"]=i
        metadata["page_content"]=len(doc.page_content)
        metadatas.append(metadata)

        document_content.append(doc.page_content)

        embeds.append(embed.tolist())
    try:
        client.add(
            ids=ids,
            embeddings=embeds,
            documents=document_content,
            metadatas=metadatas,
        )
        print("Addition successfull")
        print(f"/n Total : {client.count()}")
    except Exception as e:
        print(f"Error while addition to chroma DB: {e}")

chroma=embeddingAddition(client,chunks,embeddings)


Addition successfull
/n Total : 1176


Converting query into embedding and performing similarity search

In [31]:
def search(client,query,top_k=5):
    query_embed=conversion([query])[0]
    print(f"QUERY: {query_embed.tolist()}")
    results=client.query(query_embeddings=[query_embed.tolist()],n_results=top_k)
    return results

results=search(client,"What does clause number 250 of Bharatiya Nyaya sanhita pdf states")
results

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1733.33it/s]


embedding model dimension : <bound method SentenceTransformer.get_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)>


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.56it/s]


QUERY: [0.025359157472848892, -0.008016571402549744, -0.06243162229657173, -0.053803011775016785, -0.0002754665620159358, 0.038496460765600204, 0.0002862557885237038, -0.008535071276128292, -0.029571184888482094, -0.015317857265472412, -0.002826592419296503, -0.01854289136826992, -0.0485219806432724, -0.010634656064212322, -0.020127849653363228, 0.06456363201141357, -0.033755041658878326, -0.0058503164909780025, 0.013280433602631092, 0.04005785658955574, 0.06200925633311272, 0.043312568217515945, -0.01705012656748295, -0.026018118485808372, 0.05041823536157608, -0.031468745321035385, 0.033942099660634995, -0.005331595428287983, 0.05058975890278816, -0.011075139045715332, -0.01386046502739191, 0.1058928593993187, -0.004687462467700243, 0.024614272639155388, 0.048071809113025665, -0.013023251667618752, 0.054118067026138306, -0.0651170089840889, 0.02669386751949787, 0.014665534719824791, 0.046551529318094254, 0.004466165788471699, 0.02751864679157734, 0.028717581182718277, 0.0400690622627

{'ids': [['doc_#13_0',
   'doc_#3247_294',
   'doc_#3379_306',
   'doc_#10782_979',
   'doc_#11156_1013']],
 'embeddings': None,
 'documents': [['THE BHARATIYA NYAYA SANHITA, 2023\nNO. 45 OF 2023\n[25th December, 2023.]\nAn Act to consolidate and amend the provisions relating to offences and for\nmatters connected therewithor incidentalthereto.\nBE it enacted by Parliament in the Seventy-fourth Year of the Republic of India as\nfollows:––\nCHAPTERI\nPRELIMINARY\n1. (1) ThisAct maybe called the Bharatiya Nyaya Sanhita, 2023.\n(2) It shall come intoforce on such date as the Central Government may, bynotification\nin the Official Gazette, appoint, and different dates maybe appointed for different provisions\nof this Sanhita.\nShort title,\ncommencement\nand\napplication.\nvlk/kkj.k\nEXTRAORDINARY\nHkkx II — [k.M 1\nPART II — Section 1\nizkf/kdkj ls izdkf\'kr\nPUBLISHED BY AUTHORITY\nlañ 53]\nubZ fnYyh] lkseokj] fnlEcj 25] 2023@ikS"k 4] 1945 ¼\'kd½\nNo. 53]\nNEW DELHI, MONDAY, DECEMBER 25,

In [29]:
results["metadatas"][0]
    

[{'creationdate': '2023-06-28T10:58:56+02:00',
  'author': '',
  'creator': 'Online2PDF.com',
  'moddate': '',
  'title': '',
  'modDate': '',
  'index': 1013,
  'source': 'data\\pds\\ipc.pdf',
  'subject': '',
  'format': 'PDF 1.4',
  'page_content': 204,
  'trapped': '',
  'file_path': 'data\\pds\\ipc.pdf',
  'creationDate': "D:20230628105856+02'00'",
  'keywords': '',
  'producer': 'Online2PDF.com',
  'total_pages': 119,
  'page': 90},
 {'subject': '',
  'total_pages': 119,
  'modDate': '',
  'creationDate': "D:20230628105856+02'00'",
  'page': 13,
  'file_path': 'data\\pds\\ipc.pdf',
  'title': '',
  'page_content': 906,
  'creationdate': '2023-06-28T10:58:56+02:00',
  'moddate': '',
  'keywords': '',
  'creator': 'Online2PDF.com',
  'source': 'data\\pds\\ipc.pdf',
  'producer': 'Online2PDF.com',
  'format': 'PDF 1.4',
  'index': 591,
  'trapped': '',
  'author': ''},
 {'file_path': 'data\\pds\\ipc.pdf',
  'moddate': '',
  'format': 'PDF 1.4',
  'modDate': '',
  'creator': 'Online2